# vLLM Checkpoint Export

This notebook converts trained LoRA runs into deployment checkpoints used by the dedicated publication evaluators. It merges each adapter into its BF16 base model first, then optionally creates independent vLLM-compatible FP8, NVIDIA ModelOpt NVFP4 W4A4, or weight-only NVFP4A16 exports.

**Run cells from top to bottom.** Edit only Cell 2 for normal use. Cell 4 performs all exports in isolated subprocesses and records manifests, quantization metadata, hashes, sizes, timings, and errors on Drive.

Important constraints:

- FP8 uses LLM Compressor `FP8_DYNAMIC` by default; `FP8_BLOCK` is also selectable per export.
- Native NVFP4 uses NVIDIA ModelOpt whole-model calibration with packed E2M1 FP4 weights and dynamic FP4 activations.
- Merge BF16 base weights and the BF16 LoRA delta before quantization. Do not add an adapter directly to packed quantized weights.
- Use `VLLM_Publication_Evaluation_Colab.ipynb` for quality and `VLLM_Parallelism_Benchmark_Colab.ipynb` for throughput and latency. This notebook does not publish evaluation metrics.


In [ ]:
# Cell 1: mount Drive, install export dependencies, and stage project code locally
import os
import shutil
import subprocess
import sys

from google.colab import drive, userdata

drive.mount('/content/drive')

DRIVE_BUNDLE_DIR = '/content/drive/MyDrive/colab_drive_bundle'
NESTED_PACKAGE_DIR = f'{DRIVE_BUNDLE_DIR}/fen_move_colab'
PACKAGE_ON_DRIVE = (
    NESTED_PACKAGE_DIR
    if os.path.isfile(f'{NESTED_PACKAGE_DIR}/__init__.py')
    else DRIVE_BUNDLE_DIR
)
WORKSPACE = '/content/fen_move_vllm_workspace'

if not os.path.isfile(f'{PACKAGE_ON_DRIVE}/__init__.py'):
    raise FileNotFoundError(
        f'No fen_move_colab package found in {DRIVE_BUNDLE_DIR}. Expected __init__.py '
        'either directly in the bundle or inside bundle/fen_move_colab.'
    )

# Export uses LLM Compressor for FP8 and NVIDIA ModelOpt for W4A4 NVFP4.
# The dedicated evaluation notebooks install their own pinned vLLM serving stack.
def pip_install(*packages):
    command = [sys.executable, '-m', 'pip', 'install', '--no-cache-dir', *packages]
    print('Running:', ' '.join(command), flush=True)
    subprocess.check_call(command)

pip_install('--upgrade', 'pip', 'wheel', 'setuptools<81')
subprocess.run([
    sys.executable, '-m', 'pip', 'uninstall', '-y',
    'vllm', 'llmcompressor', 'compressed-tensors', 'torchao', 'nvidia-modelopt',
], check=False)
pip_install(
    '--upgrade',
    'llmcompressor==0.12.0.1',
    'compressed-tensors==0.17.1',
    'torchao==0.17.0', 'nvidia-modelopt>=0.45,<0.47', 'transformers==5.10.1',
    'accelerate', 'datasets', 'safetensors', 'peft', 'python-chess', 'pandas', 'pyyaml', 'nvidia-ml-py',
)
subprocess.check_call([
    sys.executable, '-c',
    (
        'import torch, torchao, transformers, llmcompressor, compressed_tensors, modelopt; '
        'print(\"Export stack:\", torch.__version__, transformers.__version__, '
        'llmcompressor.__version__, compressed_tensors.__version__, torchao.__version__, modelopt.__version__)'
    ),
])

shutil.rmtree(WORKSPACE, ignore_errors=True)
os.makedirs(WORKSPACE, exist_ok=True)
# The uploaded bundle is normally flat. Keep large datasets, runs, and raw files on Drive.
shutil.copytree(
    PACKAGE_ON_DRIVE,
    f'{WORKSPACE}/fen_move_colab',
    ignore=shutil.ignore_patterns(
        'raw', 'datasets', 'runs', 'vllm_exports', 'vllm_benchmarks',
        '__pycache__', '.ipynb_checkpoints',
    ),
)
os.chdir(WORKSPACE)
if WORKSPACE not in sys.path:
    sys.path.insert(0, WORKSPACE)

try:
    token = userdata.get('HF_TOKEN') or userdata.get('hf_token')
except Exception:
    token = None
if token:
    os.environ['HF_TOKEN'] = token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = token

subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,compute_cap', '--format=csv,noheader'])
print(f'Ready. Drive bundle: {DRIVE_BUNDLE_DIR}')
print(f'Package layout detected: {PACKAGE_ON_DRIVE}')
print(f'Local project package: {WORKSPACE}/fen_move_colab')


In [ ]:
# Cell 2: EDIT THIS CELL - model export specifications
EXPORT_ROOT = f'{DRIVE_BUNDLE_DIR}/vllm_exports'
BENCHMARK_ROOT = f'{DRIVE_BUNDLE_DIR}/vllm_benchmarks'
DEFAULT_VALIDATION_PATH = f'{DRIVE_BUNDLE_DIR}/datasets/lichess_legal_v1/validation.jsonl'

# One entry per trained run. checkpoint supports:
#   'best'  -> best adapter in metrics/sft/checkpoint_evaluation.json
#   'final' -> latest adapter in state/sft_state.json
#   60000, 'checkpoint-1875', a step/checkpoint-relative path, or an absolute path
# formats may contain: 'bf16', 'fp8', 'nvfp4', 'nvfp4a16'.
EXPORT_SPECS = [
    {
        'name': 'e4b_r32_best',
        'run_name': 'gemma4_e4b_all_legal_moves_v3',
        'checkpoint': 'best',
        'formats': ['bf16', 'fp8', 'nvfp4'],
        'fp8_scheme': 'FP8_DYNAMIC',  # portable default; FP8_BLOCK is also supported
    },
    {
        'name': 'e2b_r32_best',
        'run_name': 'gemma4_e2b_all_legal_moves_v3',
        'checkpoint': 'best',
        'formats': ['bf16', 'fp8', 'nvfp4'],
        'fp8_scheme': 'FP8_DYNAMIC',  # portable default; FP8_BLOCK is also supported
    },
    {
        'name': '270m_r32_best',
        'run_name': 'gemma3_270m_all_legal_moves_v3',
        'checkpoint': 'best',
        'formats': ['bf16', 'fp8', 'nvfp4'],
        'fp8_scheme': 'FP8_DYNAMIC',  # portable default; FP8_BLOCK is also supported
    },
    {
        'name': 'e4b_r16_best',
        'run_name': 'gemma4_e4b_all_legal_moves_v4',
        'checkpoint': 'best',
        'formats': ['bf16', 'fp8', 'nvfp4'],
        'fp8_scheme': 'FP8_DYNAMIC',  # portable default; FP8_BLOCK is also supported
    },
    {
        'name': 'e2b_r16_best',
        'run_name': 'gemma4_e2b_all_legal_moves_v4',
        'checkpoint': 'best',
        'formats': ['bf16', 'fp8', 'nvfp4'],
        'fp8_scheme': 'FP8_DYNAMIC',  # portable default; FP8_BLOCK is also supported
    },
    {
        'name': '270m_r16_best',
        'run_name': 'gemma3_270m_all_legal_moves_v4',
        'checkpoint': 'best',
        'formats': ['bf16', 'fp8', 'nvfp4'],
        'fp8_scheme': 'FP8_DYNAMIC',  # portable default; FP8_BLOCK is also supported
    },
]

OVERWRITE_EXPORTS = False
MAX_SHARD_SIZE = '5GB'
NVFP4_CALIBRATION_SAMPLES = 512
NVFP4_CALIBRATION_MAX_SEQ_LENGTH = 512
NVFP4_CALIBRATION_BATCH_SIZE = 8

print(f'Exports: {len(EXPORT_SPECS)} model specification(s)')


In [ ]:
# Cell 3: export helper implementation (normally no edits needed)
import json
import os
import re
import subprocess
import sys

EXPORT_WORKER_SOURCE = r"""
import gc, json, os, re, shutil, sys, time, traceback
from pathlib import Path

def load_json(path, default=None):
    if not path or not os.path.isfile(path):
        return {} if default is None else default
    with open(path, encoding='utf-8') as handle:
        return json.load(handle)

def save_json(path, payload):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'w', encoding='utf-8') as handle:
        json.dump(payload, handle, indent=2)

def step_number(path):
    match = re.search(r'step_(\d+)', str(path))
    return int(match.group(1)) if match else -1

def checkpoint_number(path):
    match = re.search(r'checkpoint-(\d+)', str(path))
    return int(match.group(1)) if match else -1

def latest_run_config(run_root):
    paths = list(Path(run_root).glob('metrics/sft/step_*_config.json'))
    if not paths:
        raise FileNotFoundError(f'No saved SFT config snapshots under {run_root}/metrics/sft')
    return str(max(paths, key=lambda p: (step_number(p), p.stat().st_mtime)))

def adapter_dirs(run_root):
    root = Path(run_root) / 'checkpoints' / 'sft'
    return sorted(
        {p.parent for p in root.rglob('adapter_config.json')},
        key=lambda p: (step_number(p), checkpoint_number(p), p.stat().st_mtime),
    )

def relocate_checkpoint(path, run_root):
    candidate = Path(str(path)).expanduser()
    if (candidate / 'adapter_config.json').is_file():
        return str(candidate)
    parts = candidate.parts
    if 'checkpoints' in parts:
        suffix = Path(*parts[parts.index('checkpoints') + 2:])
        moved = Path(run_root) / 'checkpoints' / 'sft' / suffix
        if (moved / 'adapter_config.json').is_file():
            return str(moved)
    matches = [p for p in adapter_dirs(run_root) if p.name == candidate.name]
    if len(matches) == 1:
        return str(matches[0])
    raise FileNotFoundError(f'Adapter checkpoint does not exist and could not be relocated: {path}')

def resolve_checkpoint(spec, run_root):
    requested = spec.get('checkpoint', 'best')
    if isinstance(requested, int) or (isinstance(requested, str) and requested.isdigit()):
        sample_step = int(requested)
        matches = [p for p in adapter_dirs(run_root) if step_number(p) == sample_step]
        if not matches:
            raise FileNotFoundError(f'No adapters found for step_{sample_step} in {run_root}')
        return str(max(matches, key=lambda p: (checkpoint_number(p), p.stat().st_mtime)))
    if str(requested).lower() == 'best':
        report = load_json(os.path.join(run_root, 'metrics', 'sft', 'checkpoint_evaluation.json'))
        path = report.get('best', {}).get('checkpoint')
        if not path:
            raise FileNotFoundError('Best checkpoint requested, but checkpoint_evaluation.json has no best checkpoint.')
        return relocate_checkpoint(path, run_root)
    if str(requested).lower() == 'final':
        state = load_json(os.path.join(run_root, 'state', 'sft_state.json'))
        path = state.get('latest_checkpoint')
        if not path:
            raise FileNotFoundError('Final checkpoint requested, but sft_state.json has no latest_checkpoint.')
        return relocate_checkpoint(path, run_root)
    candidate = Path(str(requested))
    if not candidate.is_absolute():
        direct = Path(run_root) / 'checkpoints' / 'sft' / candidate
        if (direct / 'adapter_config.json').is_file():
            return str(direct)
    return relocate_checkpoint(str(candidate), run_root)

def base_model_for_adapter(adapter_path, fallback=None):
    cfg = load_json(os.path.join(adapter_path, 'adapter_config.json'))
    model = cfg.get('base_model_name_or_path') or fallback
    if not model:
        raise ValueError(f'Could not resolve base model for {adapter_path}')
    return str(model)

def release(model=None, tokenizer=None):
    if model is not None:
        del model
    if tokenizer is not None:
        del tokenizer
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
    except Exception:
        pass

def directory_size(path):
    return sum(p.stat().st_size for p in Path(path).rglob('*') if p.is_file())

def compression_metadata(path):
    config = load_json(os.path.join(path, 'config.json'))
    metadata = config.get('quantization_config') or config.get('compression_config')
    if not metadata:
        raise RuntimeError(f'No quantization metadata found in {path}/config.json')
    return metadata

def has_input_activation_quantization(value):
    if isinstance(value, dict):
        if 'input_activations' in value and value['input_activations'] is not None:
            return True
        return any(has_input_activation_quantization(item) for item in value.values())
    if isinstance(value, list):
        return any(has_input_activation_quantization(item) for item in value)
    return False

def load_bf16_model(path):
    import torch
    from transformers import AutoModelForCausalLM
    return AutoModelForCausalLM.from_pretrained(
        path,
        dtype=torch.bfloat16,
        device_map='auto',
        low_cpu_mem_usage=True,
        trust_remote_code=True,
    )

def load_nvfp4_model(path, llmcompressor_context=False):
    import torch
    from transformers import AutoConfig, AutoModelForCausalLM
    config = AutoConfig.from_pretrained(path, trust_remote_code=True)
    if getattr(config, 'model_type', None) == 'gemma4':
        from transformers import Gemma4ForConditionalGeneration
        load_kwargs = dict(
            dtype=torch.bfloat16, device_map='auto',
            low_cpu_mem_usage=True, trust_remote_code=True,
        )
        if llmcompressor_context:
            from llmcompressor.utils import load_context
            with load_context(Gemma4ForConditionalGeneration):
                return Gemma4ForConditionalGeneration.from_pretrained(path, **load_kwargs)
        return Gemma4ForConditionalGeneration.from_pretrained(path, **load_kwargs)
    return AutoModelForCausalLM.from_pretrained(
        path, dtype=torch.bfloat16, device_map='auto',
        low_cpu_mem_usage=True, trust_remote_code=True,
    )

def merge_adapter(adapter_path, base_model, output_dir, max_shard_size):
    import torch
    from peft import PeftModel
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)
    base = AutoModelForCausalLM.from_pretrained(
        base_model,
        dtype=torch.bfloat16,
        device_map='auto',
        low_cpu_mem_usage=True,
        trust_remote_code=True,
    )
    peft_model = PeftModel.from_pretrained(base, adapter_path, is_trainable=False)
    merged = peft_model.merge_and_unload(safe_merge=True)
    os.makedirs(output_dir, exist_ok=True)
    merged.save_pretrained(
        output_dir,
        safe_serialization=True,
        max_shard_size=max_shard_size,
    )
    tokenizer.save_pretrained(output_dir)
    release(merged, tokenizer)

def quantize_fp8(merged_dir, output_dir, scheme, max_shard_size):
    from llmcompressor import model_free_ptq, oneshot
    from llmcompressor.modifiers.quantization import QuantizationModifier
    from transformers import AutoTokenizer
    ignore = ['re:.*vision.*', 're:.*audio.*', 'lm_head', 're:.*embed.*']
    normalized = str(scheme).upper()
    if normalized == 'FP8_BLOCK':
        model_free_ptq(
            model_stub=merged_dir,
            save_directory=output_dir,
            scheme='FP8_BLOCK',
            ignore=ignore,
            max_workers=max(1, min(8, os.cpu_count() or 1)),
            device='cuda:0',
        )
        tokenizer = AutoTokenizer.from_pretrained(merged_dir, trust_remote_code=True)
        tokenizer.save_pretrained(output_dir)
        return
    if normalized != 'FP8_DYNAMIC':
        raise ValueError("fp8_scheme must be 'FP8_DYNAMIC' or 'FP8_BLOCK'.")
    tokenizer = AutoTokenizer.from_pretrained(merged_dir, trust_remote_code=True)
    model = load_bf16_model(merged_dir)
    recipe = QuantizationModifier(targets='Linear', scheme='FP8_DYNAMIC', ignore=ignore)
    oneshot(model=model, processor=tokenizer, recipe=recipe)
    os.makedirs(output_dir, exist_ok=True)
    model.save_pretrained(output_dir, save_compressed=True, max_shard_size=max_shard_size)
    tokenizer.save_pretrained(output_dir)
    release(model, tokenizer)

def modelopt_calibration_texts(merged_dir, calibration_path, num_samples):
    from transformers import AutoTokenizer
    if not calibration_path or not os.path.isfile(calibration_path):
        raise FileNotFoundError(f'NVFP4 calibration dataset not found: {calibration_path}')
    texts = []
    with open(calibration_path, encoding='utf-8') as handle:
        for line in handle:
            if not line.strip():
                continue
            row = json.loads(line)
            instruction = row.get('instruction')
            output = row.get('output')
            if instruction and output:
                texts.append(
                    f"<start_of_turn>user\n{instruction}<end_of_turn>\n"
                    f"<start_of_turn>model\n{output}<end_of_turn>"
                )
            if len(texts) >= int(num_samples):
                break
    if not texts:
        raise ValueError(f'No instruction/output records found in {calibration_path}')
    tokenizer = AutoTokenizer.from_pretrained(merged_dir, trust_remote_code=True)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer, texts

def register_modelopt_gemma4_experts():
    import torch
    import torch.nn as nn
    from modelopt.torch.quantization.nn import QuantModule, QuantModuleRegistry
    try:
        from transformers.models.gemma4.modeling_gemma4 import Gemma4TextExperts
    except ImportError:
        return False
    if Gemma4TextExperts in getattr(QuantModuleRegistry, '_registry', {}):
        return False

    class _Gemma4ExpertModule(nn.Module):
        def __init__(self, hidden_dim, intermediate_dim):
            super().__init__()
            self.gate_proj = nn.Linear(hidden_dim, intermediate_dim, bias=False)
            self.up_proj = nn.Linear(hidden_dim, intermediate_dim, bias=False)
            self.down_proj = nn.Linear(intermediate_dim, hidden_dim, bias=False)

    class _QuantGemma4TextExperts(QuantModule):
        def _setup(self):
            from accelerate import init_empty_weights
            dtype, device = self.gate_up_proj.dtype, self.gate_up_proj.device

            def copy_weight(module, weight):
                module.to_empty(device=device)
                with torch.no_grad():
                    module.weight.data = weight.detach().to(dtype=dtype, device=device)

            expert_dim = self.intermediate_dim
            with init_empty_weights():
                experts = nn.ModuleList([
                    _Gemma4ExpertModule(self.hidden_dim, expert_dim)
                    for _ in range(self.num_experts)
                ])
            for index in range(self.num_experts):
                copy_weight(experts[index].gate_proj, self.gate_up_proj[index, :expert_dim, :])
                copy_weight(experts[index].up_proj, self.gate_up_proj[index, expert_dim:, :])
                copy_weight(experts[index].down_proj, self.down_proj[index])
            delattr(self, 'gate_up_proj')
            delattr(self, 'down_proj')
            for index, expert in enumerate(experts):
                self.add_module(str(index), expert)

        def __len__(self):
            return self.num_experts

        def __iter__(self):
            for index in range(self.num_experts):
                yield getattr(self, str(index))

        def __getitem__(self, index):
            return getattr(self, str(int(index)))

        def forward(self, hidden_states, top_k_index, top_k_weights):
            final_hidden_states = torch.zeros_like(hidden_states)
            with torch.no_grad():
                expert_mask = torch.nn.functional.one_hot(
                    top_k_index, num_classes=self.num_experts
                ).permute(2, 1, 0)
                expert_hit = torch.greater(expert_mask.sum(dim=(-1, -2)), 0).nonzero()
            for expert_index in expert_hit.flatten():
                if int(expert_index) == self.num_experts:
                    continue
                with torch.no_grad():
                    top_k_pos, token_index = torch.where(expert_mask[expert_index])
                current = hidden_states[token_index]
                expert = self[expert_index]
                current = self.act_fn(expert.gate_proj(current)) * expert.up_proj(current)
                current = expert.down_proj(current)
                current = current * top_k_weights[token_index, top_k_pos, None]
                final_hidden_states.index_add_(
                    0, token_index, current.to(final_hidden_states.dtype)
                )
            return final_hidden_states

    QuantModuleRegistry.register(
        {Gemma4TextExperts: 'hf.Gemma4TextExperts'}
    )(_QuantGemma4TextExperts)
    return True

def fix_modelopt_gemma4_expert_keys(output_dir):
    from safetensors import safe_open
    from safetensors.torch import load_file, save_file
    files = sorted(Path(output_dir).glob('*.safetensors'))
    renamed_total = 0
    weight_map = {}
    total_size = 0
    for path in files:
        tensors = load_file(str(path), device='cpu')
        renamed = {}
        for key, tensor in tensors.items():
            new_key = key
            if '.moe.experts.' not in key:
                new_key = re.sub(r'\.experts\.(\d+)\.', r'.moe.experts.\1.', key)
            renamed_total += int(new_key != key)
            renamed[new_key] = tensor
            weight_map[new_key] = path.name
            total_size += tensor.numel() * tensor.element_size()
        if any(key not in tensors for key in renamed):
            temp_path = str(path) + '.tmp'
            save_file(renamed, temp_path)
            os.replace(temp_path, path)
        del tensors, renamed
    index_path = os.path.join(output_dir, 'model.safetensors.index.json')
    if os.path.isfile(index_path):
        save_json(index_path, {'metadata': {'total_size': total_size}, 'weight_map': weight_map})
    return renamed_total

def modelopt_nvfp4_tensor_summary(path):
    from safetensors import safe_open
    summary = {
        'packed_weight_tensors': 0, 'weight_scale_tensors': 0, 'input_scale_tensors': 0,
        'attention_packed_weight_tensors': 0, 'mlp_packed_weight_tensors': 0,
        'expert_packed_weight_tensors': 0,
    }
    for shard in Path(path).glob('*.safetensors'):
        with safe_open(str(shard), framework='pt', device='cpu') as handle:
            for key in handle.keys():
                dtype = str(handle.get_slice(key).get_dtype()).upper()
                if key.endswith('.weight') and dtype in {'U8', 'I8'}:
                    summary['packed_weight_tensors'] += 1
                    if '.self_attn.' in key or '.attention.' in key:
                        summary['attention_packed_weight_tensors'] += 1
                    if '.mlp.' in key or '.moe.' in key or '.experts.' in key:
                        summary['mlp_packed_weight_tensors'] += 1
                    if '.experts.' in key or '.moe.experts.' in key:
                        summary['expert_packed_weight_tensors'] += 1
                if 'weight_scale' in key:
                    summary['weight_scale_tensors'] += 1
                if 'input_scale' in key:
                    summary['input_scale_tensors'] += 1
    return summary

def checkpoint_declares_moe(path):
    config = load_json(os.path.join(path, 'config.json'))

    def walk(value):
        if isinstance(value, dict):
            for key, child in value.items():
                if key in {'num_experts', 'num_local_experts'}:
                    try:
                        if int(child) > 1:
                            return True
                    except (TypeError, ValueError):
                        pass
                if walk(child):
                    return True
        elif isinstance(value, list):
            return any(walk(child) for child in value)
        return False

    return walk(config)

def is_modelopt_w4a4_nvfp4(metadata):
    if not isinstance(metadata, dict):
        return False
    algo = str(metadata.get('quant_algo') or metadata.get('quantization', {}).get('quant_algo') or '').upper()
    method = str(metadata.get('quant_method') or '').lower()
    return algo == 'NVFP4' and ('modelopt' in method or method == '')

def disable_modelopt_quantizer_patterns(quant_cfg, patterns):
    rules = quant_cfg.get('quant_cfg')
    if isinstance(rules, list):
        rules.extend(
            {'quantizer_name': pattern, 'enable': False}
            for pattern in patterns
        )
        return 'ordered_list'
    if isinstance(rules, dict):
        for pattern in patterns:
            rules[pattern] = {'enable': False}
        return 'legacy_mapping'
    raise TypeError(
        'Unsupported ModelOpt quant_cfg representation: '
        f'{type(rules).__name__}'
    )

def quantize_nvfp4(merged_dir, output_dir, calibration_path, num_samples, max_seq_length, batch_size, max_shard_size):
    import copy
    import torch
    import modelopt.torch.quantization as mtq
    from modelopt.torch.export import export_hf_checkpoint
    from torch.utils.data import DataLoader
    tokenizer, texts = modelopt_calibration_texts(merged_dir, calibration_path, num_samples)
    registered_expert_plugin = register_modelopt_gemma4_experts()
    model = load_nvfp4_model(merged_dir)
    model.eval()
    quant_cfg = copy.deepcopy(mtq.NVFP4_DEFAULT_CFG)
    exclusion_api = disable_modelopt_quantizer_patterns(quant_cfg, (
        '*vision*', '*audio*', '*embed_vision*', '*embed_audio*',
        '*multi_modal_projector*', '*lm_head*', '*embed_tokens*', '*router*',
    ))

    def collate(batch):
        return tokenizer(
            batch, return_tensors='pt', padding=True, truncation=True,
            max_length=int(max_seq_length), add_special_tokens=False,
        )

    loader = DataLoader(texts, batch_size=int(batch_size), shuffle=False, collate_fn=collate)

    def forward_loop(active_model):
        device = active_model.get_input_embeddings().weight.device
        with torch.inference_mode():
            for batch in loader:
                batch = {key: value.to(device) for key, value in batch.items()}
                active_model(**batch, use_cache=False)

    model = mtq.quantize(model, quant_cfg, forward_loop=forward_loop)
    os.makedirs(output_dir, exist_ok=True)
    export_hf_checkpoint(model, dtype=torch.bfloat16, export_dir=output_dir)
    tokenizer.save_pretrained(output_dir)
    renamed_keys = fix_modelopt_gemma4_expert_keys(output_dir)
    release(model, tokenizer)
    return {
        'samples': len(texts),
        'batch_size': int(batch_size),
        'quant_cfg_representation': exclusion_api,
        'registered_gemma4_expert_plugin': registered_expert_plugin,
        'renamed_expert_keys': renamed_keys,
    }

def quantize_nvfp4a16(merged_dir, output_dir, max_shard_size):
    from llmcompressor import oneshot
    from llmcompressor.modifiers.quantization import QuantizationModifier
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(merged_dir, trust_remote_code=True)
    model = load_nvfp4_model(merged_dir, llmcompressor_context=True)
    recipe = QuantizationModifier(
        targets='Linear',
        scheme='NVFP4A16',
        ignore=[
            'lm_head',
            're:.*vision_tower.*',
            're:.*audio_tower.*',
            're:.*embed_vision.*',
            're:.*embed_audio.*',
            're:.*embed_tokens.*',
        ],
    )
    oneshot(model=model, recipe=recipe)
    os.makedirs(output_dir, exist_ok=True)
    model.save_pretrained(output_dir, save_compressed=True, max_shard_size=max_shard_size)
    tokenizer.save_pretrained(output_dir)
    release(model, tokenizer)

def main(spec_path):
    payload = load_json(spec_path)
    spec = payload['spec']
    settings = payload['settings']
    name = spec['name']
    export_dir = os.path.join(settings['export_root'], name)
    manifest_path = os.path.join(export_dir, 'export_manifest.json')
    manifest = {
        'name': name,
        'status': 'running',
        'requested_formats': [str(x).lower() for x in spec.get('formats', ['bf16'])],
        'formats': {},
        'started_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    }
    save_json(manifest_path, manifest)
    try:
        run_name = spec.get('run_name')
        run_root = os.path.join(settings['drive_bundle_dir'], 'runs', run_name) if run_name else None
        run_config_path = latest_run_config(run_root) if run_root else None
        run_config = load_json(run_config_path)
        source_model = spec.get('source_model')
        adapter_path = None
        if source_model:
            base_model = str(source_model)
        else:
            if not run_root or not os.path.isdir(run_root):
                raise FileNotFoundError(f'Run directory not found: {run_root}')
            adapter_path = resolve_checkpoint(spec, run_root)
            base_model = base_model_for_adapter(adapter_path, run_config.get('sft', {}).get('model_id'))
        calibration_path = spec.get('calibration_path') or run_config.get('data', {}).get('sft_output')

        manifest.update({
            'run_name': run_name,
            'run_root': run_root,
            'run_config_path': run_config_path,
            'adapter_checkpoint': adapter_path,
            'base_model': base_model,
            'calibration_training_path': calibration_path,
            'training_precision_mode': run_config.get('precision', {}).get('mode'),
            'training_output_mode': run_config.get('sft', {}).get('output_mode'),
            'training_lora': run_config.get('lora'),
        })
        os.makedirs(export_dir, exist_ok=True)
        merged_dir = os.path.join(export_dir, 'merged_bf16')
        formats = manifest['requested_formats']
        if any(fmt not in {'bf16', 'fp8', 'nvfp4', 'nvfp4a16'} for fmt in formats):
            raise ValueError(f'Unsupported export format in {formats}')

        if settings['overwrite_exports'] and os.path.isdir(export_dir):
            for child in Path(export_dir).iterdir():
                if child.name != 'export_manifest.json':
                    shutil.rmtree(child) if child.is_dir() else child.unlink()

        if not os.path.isfile(os.path.join(merged_dir, 'config.json')):
            started = time.perf_counter()
            if adapter_path:
                merge_adapter(adapter_path, base_model, merged_dir, settings['max_shard_size'])
            else:
                model = load_bf16_model(base_model)
                from transformers import AutoTokenizer
                tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)
                os.makedirs(merged_dir, exist_ok=True)
                model.save_pretrained(merged_dir, safe_serialization=True, max_shard_size=settings['max_shard_size'])
                tokenizer.save_pretrained(merged_dir)
                release(model, tokenizer)
            manifest['formats']['bf16'] = {
                'status': 'completed', 'path': merged_dir,
                'seconds': time.perf_counter() - started,
                'size_bytes': directory_size(merged_dir),
                'quantization': None,
            }
            save_json(manifest_path, manifest)
        else:
            manifest['formats']['bf16'] = {
                'status': 'reused', 'path': merged_dir,
                'size_bytes': directory_size(merged_dir), 'quantization': None,
            }

        if 'fp8' in formats:
            scheme = str(spec.get('fp8_scheme', 'FP8_DYNAMIC')).upper()
            folder = 'fp8_block' if scheme == 'FP8_BLOCK' else 'fp8_dynamic'
            target = os.path.join(export_dir, folder)
            try:
                if os.path.isfile(os.path.join(target, 'config.json')) and not settings['overwrite_exports']:
                    metadata = compression_metadata(target)
                    status, seconds = 'reused', 0.0
                else:
                    shutil.rmtree(target, ignore_errors=True)
                    started = time.perf_counter()
                    quantize_fp8(merged_dir, target, scheme, settings['max_shard_size'])
                    seconds = time.perf_counter() - started
                    metadata = compression_metadata(target)
                    status = 'completed'
                manifest['formats']['fp8'] = {
                    'status': status, 'path': target, 'scheme': scheme,
                    'seconds': seconds, 'size_bytes': directory_size(target),
                    'quantization': metadata,
                }
            except Exception as exc:
                manifest['formats']['fp8'] = {
                    'status': 'failed', 'error_type': type(exc).__name__, 'error': str(exc),
                    'traceback': traceback.format_exc(),
                }
            save_json(manifest_path, manifest)
            release()

        if 'nvfp4' in formats:
            target = os.path.join(export_dir, 'nvfp4')
            try:
                reusable = os.path.isfile(os.path.join(target, 'config.json')) and not settings['overwrite_exports']
                if reusable:
                    metadata = compression_metadata(target)
                    reusable = is_modelopt_w4a4_nvfp4(metadata)
                if reusable:
                    status, seconds = 'reused', 0.0
                    calibration_info = {
                        'samples': int(settings['nvfp4_calibration_samples']),
                        'batch_size': int(settings['nvfp4_calibration_batch_size']),
                    }
                else:
                    shutil.rmtree(target, ignore_errors=True)
                    started = time.perf_counter()
                    calibration_info = quantize_nvfp4(
                        merged_dir, target, calibration_path,
                        settings['nvfp4_calibration_samples'],
                        settings['nvfp4_calibration_max_seq_length'],
                        settings['nvfp4_calibration_batch_size'],
                        settings['max_shard_size'],
                    )
                    seconds = time.perf_counter() - started
                    metadata = compression_metadata(target)
                    status = 'completed'
                tensor_summary = modelopt_nvfp4_tensor_summary(target)
                if not is_modelopt_w4a4_nvfp4(metadata):
                    raise RuntimeError(
                        'NVFP4 export is not a ModelOpt quant_algo=NVFP4 checkpoint; refusing to label it W4A4.'
                    )
                required_counts = (
                    'packed_weight_tensors', 'weight_scale_tensors', 'input_scale_tensors',
                    'attention_packed_weight_tensors', 'mlp_packed_weight_tensors',
                )
                missing_counts = [key for key in required_counts if tensor_summary[key] <= 0]
                if checkpoint_declares_moe(merged_dir) and tensor_summary['expert_packed_weight_tensors'] <= 0:
                    missing_counts.append('expert_packed_weight_tensors')
                if missing_counts:
                    raise RuntimeError(
                        'NVFP4 export is incomplete; missing packed/scaled tensor classes '
                        f'{missing_counts}: {tensor_summary}'
                    )
                manifest['formats']['nvfp4'] = {
                    'status': status, 'path': target, 'scheme': 'NVFP4',
                    'seconds': seconds, 'size_bytes': directory_size(target),
                    'weight_precision': 'fp4_group16_with_fp8_local_scales',
                    'activation_precision': 'fp4_dynamic_group16',
                    'calibration_required': True,
                    'calibration_path': calibration_path,
                    'producer': 'nvidia-modelopt',
                    'calibration_samples': int(calibration_info['samples']),
                    'calibration_batch_size': int(calibration_info['batch_size']),
                    'calibration_max_seq_length': int(settings['nvfp4_calibration_max_seq_length']),
                    'tensor_summary': tensor_summary,
                    'calibration_info': calibration_info,
                    'quantization': metadata,
                }
            except Exception as exc:
                shutil.rmtree(target, ignore_errors=True)
                manifest['formats']['nvfp4'] = {
                    'status': 'failed', 'error_type': type(exc).__name__, 'error': str(exc),
                    'traceback': traceback.format_exc(),
                }
            save_json(manifest_path, manifest)
            release()

        if 'nvfp4a16' in formats:
            target = os.path.join(export_dir, 'nvfp4a16')
            try:
                if os.path.isfile(os.path.join(target, 'config.json')) and not settings['overwrite_exports']:
                    metadata = compression_metadata(target)
                    status, seconds = 'reused', 0.0
                else:
                    shutil.rmtree(target, ignore_errors=True)
                    started = time.perf_counter()
                    quantize_nvfp4a16(merged_dir, target, settings['max_shard_size'])
                    seconds = time.perf_counter() - started
                    metadata = compression_metadata(target)
                    status = 'completed'
                manifest['formats']['nvfp4a16'] = {
                    'status': status, 'path': target, 'scheme': 'NVFP4A16',
                    'seconds': seconds, 'size_bytes': directory_size(target),
                    'activation_precision': 'bf16_or_fp16',
                    'calibration_required': False,
                    'quantization': metadata,
                }
            except Exception as exc:
                manifest['formats']['nvfp4a16'] = {
                    'status': 'failed', 'error_type': type(exc).__name__, 'error': str(exc),
                    'traceback': traceback.format_exc(),
                }
            save_json(manifest_path, manifest)
            release()

        manifest['status'] = 'completed_with_errors' if any(
            value.get('status') == 'failed' for value in manifest['formats'].values()
        ) else 'completed'
    except Exception as exc:
        manifest.update({
            'status': 'failed', 'error_type': type(exc).__name__, 'error': str(exc),
            'traceback': traceback.format_exc(),
        })
    manifest['finished_utc'] = time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())
    try:
        import torch, torchao, transformers, llmcompressor, compressed_tensors, modelopt
        manifest['environment'] = {
            'torch': torch.__version__, 'cuda': torch.version.cuda,
            'transformers': transformers.__version__,
            'llmcompressor': getattr(llmcompressor, '__version__', None),
            'compressed_tensors': getattr(compressed_tensors, '__version__', None),
            'torchao': getattr(torchao, '__version__', None),
            'modelopt': getattr(modelopt, '__version__', None),
            'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
            'compute_capability': list(torch.cuda.get_device_capability()) if torch.cuda.is_available() else None,
        }
    except Exception:
        pass
    save_json(manifest_path, manifest)
    print(json.dumps(manifest, indent=2), flush=True)
    raise SystemExit(0 if manifest['status'] != 'failed' else 1)

if __name__ == '__main__':
    main(sys.argv[1])
"""

def write_worker(path, content):
    with open(path, 'w', encoding='utf-8') as handle:
        handle.write(content)

def stream_process(command, env=None):
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    for line in process.stdout:
        print(line, end='')
    return process.wait()

def load_json(path, default=None):
    if not path or not os.path.isfile(path):
        return {} if default is None else default
    with open(path, encoding='utf-8') as handle:
        return json.load(handle)

def save_json(path, payload):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'w', encoding='utf-8') as handle:
        json.dump(payload, handle, indent=2)

def safe_name(value):
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', str(value)).strip('_') or 'model'

def run_exports(specs):
    os.makedirs(EXPORT_ROOT, exist_ok=True)
    names = [spec.get('name') for spec in specs]
    if any(not name for name in names) or len(set(names)) != len(names):
        raise ValueError('Every export spec needs a non-empty, unique name.')
    worker_path = '/content/fen_move_vllm_export_worker.py'
    write_worker(worker_path, EXPORT_WORKER_SOURCE)
    settings = {
        'drive_bundle_dir': DRIVE_BUNDLE_DIR,
        'export_root': EXPORT_ROOT,
        'overwrite_exports': bool(OVERWRITE_EXPORTS),
        'max_shard_size': MAX_SHARD_SIZE,
        'nvfp4_calibration_samples': int(NVFP4_CALIBRATION_SAMPLES),
        'nvfp4_calibration_max_seq_length': int(NVFP4_CALIBRATION_MAX_SEQ_LENGTH),
        'nvfp4_calibration_batch_size': int(NVFP4_CALIBRATION_BATCH_SIZE),
    }
    manifests = []
    for index, spec in enumerate(specs, start=1):
        print(f"\n===== EXPORT {index}/{len(specs)}: {spec['name']} =====", flush=True)
        spec_path = f"/content/export_{safe_name(spec['name'])}.json"
        save_json(spec_path, {'spec': spec, 'settings': settings})
        env = os.environ.copy()
        env['PYTHONPATH'] = WORKSPACE + os.pathsep + env.get('PYTHONPATH', '')
        code = stream_process([sys.executable, worker_path, spec_path], env=env)
        manifest_path = f"{EXPORT_ROOT}/{spec['name']}/export_manifest.json"
        manifest = load_json(manifest_path, {'name': spec['name'], 'status': 'failed', 'exit_code': code})
        manifest['worker_exit_code'] = code
        manifests.append(manifest)
    return manifests

print('Export helpers loaded. Run Cell 4 to merge and quantize checkpoints.')


In [ ]:
# Cell 4: EXPORT every model in EXPORT_SPECS (merged BF16 + requested quantized formats)
import pandas as pd

EXPORT_MANIFESTS = run_exports(EXPORT_SPECS)

export_summary = []
for manifest in EXPORT_MANIFESTS:
    for fmt, info in manifest.get('formats', {}).items():
        export_summary.append({
            'model': manifest.get('name'),
            'overall_status': manifest.get('status'),
            'format': fmt,
            'format_status': info.get('status'),
            'path': info.get('path'),
            'size_gib': (info.get('size_bytes') or 0) / (1024 ** 3),
            'seconds': info.get('seconds'),
            'error': info.get('error'),
        })
display(pd.DataFrame(export_summary).round(4))
print(f'Export root: {EXPORT_ROOT}')
